Tarea 4: Chunking fijo (application/services/retrieval_service.py)
Implementa la función `overlap_chunking` que recibe texto crudo y parámetros de chunking,
y devuelve una `list[Chunk]` (modelo de dominio). Chunking fijo: 500 caracteres con 50 de overlap.
Maneja el caso borde de texto más corto que chunk_size (retorna 1 solo chunk).
Escribe tests unitarios parametrizados con múltiples escenarios:
texto corto, texto largo, texto de exactamente chunk_size, y texto de chunk_size+1.

Una vez se tienen los chunks se debe modificar hacia el objeto Document que es estandar en el uso de bases vectoriales y que está declarado como el objeto genérico para el protocolo VectorStore

In [1]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [2]:
import sys

__import__("pysqlite3")
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")


In [3]:
import chromadb

from sentence_transformers import SentenceTransformer

from researchos.paths import CHROMA_DIR
from researchos.domain.models import Document
from researchos.domain.interfaces import VectorStore

from researchos.infrastructure.retrieval.embedder import LocalEmbedder
from researchos.infrastructure.retrieval.chroma import ChromaVectorStore

class ChromaVectorStore:
    def __init__(self, embedder: LocalEmbedder, collection_name: str = "papers", embedder_metadata: dict = {"hnsw:space": "cosine"}):
        self.embedder = embedder
        self.embedder_metadata = embedder_metadata
        self.client = chromadb.PersistentClient(path=str(CHROMA_DIR))
        self.collection = self.client.get_or_create_collection(name=collection_name, metadata=self.embedder_metadata)        

    async def search(self, query: str, k: int) -> list[Document]:
        """Search for the top-k most relevant documents."""

        query_embedding = self.embedder.embed(query)
        retrieved_docs = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )

        results = [
            Document(
                doc_id=id,
                text=text,
                metadata=metadata,
                score=self._distance_to_score(distance)
            )

            for id, text, metadata, distance in zip(
                retrieved_docs["ids"][0],
                retrieved_docs["documents"][0],
                retrieved_docs["metadatas"][0],
                retrieved_docs["distances"][0]
            )
        ]

        return results

    async def upsert(self, documents: list[Document]) -> None:
        """Insert or update documents in the store."""

        vectors = self.embedder.embed_batch([doc.text for doc in documents])

        self.collection.upsert(
            ids=[doc.doc_id for doc in documents],
            embeddings=vectors,
            documents=[doc.text for doc in documents],  # ← guarda el texto
            metadatas=[doc.metadata if doc.metadata else {'source': 'unknown'} for doc in documents]
        )

    def _distance_to_score(self, distance: float) -> float:
        space = self.embedder_metadata.get("hnsw:space", "cosine")
        if space == "cosine":
            return 1 - (distance / 2)
        elif space == "l2":
            return 1 / (1 + distance)
        else:
            return 1 - distance

Failed to reload module 'sqlite3' from file '/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/pysqlite3/__init__.py'
Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/.pyenv/versions/3.11.8/lib/python3.11/importlib/__init__.py", line 148, in reload
    raise ImportError(msg.format(name), name=name)
ImportError: module pysqlite3 not in sys.modules
[autoreload of sqlite3 failed: Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/exte

# Pruebas

In [4]:
embedder = LocalEmbedder()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
chroma_store = ChromaVectorStore(
    embedder=embedder,
    collection_name='papers',
    embedder_metadata={"hnsw:space": "cosine"}
)

In [6]:
docs = [
    Document(doc_id="1", text="LLM agents use reasoning to solve tasks", metadata={}),
    Document(doc_id="2", text="RAG combines retrieval with generation", metadata={}),
    Document(doc_id="3", text="Python is a programming language", metadata={}),
]

await chroma_store.upsert(docs)

In [7]:
results = await chroma_store.search("how do agents reason?", k=2)

In [8]:
for r in results:
    print(r.doc_id, r.score, r.text)

1 0.8244150876998901 LLM agents use reasoning to solve tasks
3 0.5539366900920868 Python is a programming language


In [9]:
results

[Document(doc_id='1', text='LLM agents use reasoning to solve tasks', metadata={'source': 'unknown'}, score=0.8244150876998901),
 Document(doc_id='3', text='Python is a programming language', metadata={'source': 'unknown'}, score=0.5539366900920868)]

## Papers reales

In [10]:
import os
import fitz

from researchos.paths import PAPERS_DIR
from researchos.infrastructure.retrieval.embedder import LocalEmbedder
from researchos.infrastructure.retrieval.chroma import ChromaVectorStore
from researchos.infrastructure.data.arxiv import search_papers
from researchos.application.services.ingestion_service import extract_text_pdf
from researchos.application.services.retrieval_service import overlap_chunking, chunk_to_document

# Cargar 3 PDFs locales
entries = os.listdir(PAPERS_DIR)[:3]

embedder = LocalEmbedder()
store = ChromaVectorStore(embedder=embedder, collection_name="test_papers")

for filename in entries:
    local_pdf_path = PAPERS_DIR / filename
    
    # Extraer texto
    full_text = ""
    doc = fitz.open(local_pdf_path)
    for page in doc:
        full_text += page.get_text()
    
    # Chunkear
    chunks = overlap_chunking(text=full_text, paper_id=filename)
    
    # Convertir a Document
    documents = [chunk_to_document(chunk) for chunk in chunks]
    
    # Guardar en Chroma
    await store.upsert(documents)
    print(f"Ingested {len(documents)} chunks from {filename}")

# Buscar
results = await store.search("multi-agent LLM systems", k=3)
for r in results:
    print(f"\nscore: {r.score:.3f}")
    print(f"text: {r.text[:200]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ingested 105 chunks from kamil_szczepanik_2025.pdf
Ingested 307 chunks from changxi_zhu_2022.pdf
Ingested 273 chunks from debangshu_banerjee_2026.pdf

score: 0.874
text: V006T06A044. American Soci-
ety of Mechanical Engineers.
H¨andler, T. (2023).
A taxonomy for autonomous llm-
powered multi-agent architectures.
In KMIS, pages
85–98.
Hatalis, K., Christou, D., Myers, 

score: 0.869
text: mory functions within LLM-based multi-agent
systems (Kostka and Chudziak, 2024).
2.3.1
Agent Orchestration
Along with research about the application of LLMs
in multi-agent systems, a new subdomain of 

score: 0.835
text: Hatalis et al., 2023).
6.3
Future Work
Studies have shown a promising and worth-exploring
domain of multi-agent LLM systems. Examining how
other agent orchestration architectures might perform
would p


In [11]:
results

[Document(doc_id='kamil_szczepanik_2025.pdf_104', text='V006T06A044. American Soci-\nety of Mechanical Engineers.\nH¨andler, T. (2023).\nA taxonomy for autonomous llm-\npowered multi-agent architectures.\nIn KMIS, pages\n85–98.\nHatalis, K., Christou, D., Myers, J., Jones, S., Lambert, K.,\nAmos-Binks, A., Dannenhauer, Z., and Dannenhauer,\nD. (2023).\nMemory matters: The need to improve\nlong-term memory in llm-agents. In Proceedings of\nthe AAAI Symposium Series, volume 2, pages 277–\n280.\nHu, S., Lu, C., and Clune, J. (2024). Automated design of\nagentic system', metadata={'chunk_index': 104, 'end_char': 47300, 'start_char': 46800, 'paper_id': 'kamil_szczepanik_2025.pdf', 'chunk_size': 500, 'overlap': 50}, score=0.8741506934165955),
 Document(doc_id='kamil_szczepanik_2025.pdf_32', text='mory functions within LLM-based multi-agent\nsystems (Kostka and Chudziak, 2024).\n2.3.1\nAgent Orchestration\nAlong with research about the application of LLMs\nin multi-agent systems, a new subdom